 # `huggingface-hub` API  
We add here code snippets and comments for all APIs and modules of `huggingface-hib` library.

## HF credential management
Main reference is [HF tutorial on huggingface-hub](https://huggingface.co/docs/huggingface_hub/en/quick-start#manage-multiple-tokens-locally)

1. Shared token files (all huggingface-hub functions use these files)

In [ ]:
%%bash

# "Weird" assignment: := sets HF_HOME to the passed value if unset. We use the "no-command" : of bash to prevent printing
: ${HF_HOME:=~/.cache/huggingface}

echo "${HF_HOME}/token"
cat "${HF_HOME}/token"; echo; echo

echo "${HF_HOME}/stored_tokens"
cat "${HF_HOME}/stored_tokens"

# show the name of the currently logged in user
echo $(huggingface-cli whoami)

In [28]:
from huggingface_hub import login, whoami, HFCacheInfo, HfFolder, auth_list, auth_switch, auth_check

In [ ]:
import os

# Convenience store
# 1. readonly token
rotoken = os.environ["read-only-token"]
# 2. fine-grained token
fgtoken = os.environ["fine-graoned-token"]

! huggingface-cli login --token {fgtoken}

In [16]:
# Login to HF using a token literal
login(token=rotoken)

# Now the 'token' file contains the selected token. this token will be used by HF libraries globaly on this computer!
# to disable this behavior (i.e. make HF ask for authentication each time) set HF_HUB_DISABLE_IMPLICIT_TOKEN=1
print(HfFolder.get_token() == rotoken)

# Check the names of the global functions in HfFolder (only )
print(*[o for o in dir(HfFolder) if not o.startswith('__')])

True
delete_token get_token save_token


In [29]:
# There's no direct way to get the name of the currently selected token.
# Indirectly, use whoami:
print(whoami()["auth"]["accessToken"]["displayName"])

# To get all the token ever registered with HF use the auth_list function or huggingface-cli
print(auth_list())
!huggingface-cli auth list

fine-grained-token
  name                | token          
----------------------|---------------
* fine-grained-token  | hf_****KBKQ    
  read-token          | hf_****nHdC    
None
  name                | token          
----------------------|---------------
* fine-grained-token  | hf_****KBKQ    
  read-token          | hf_****nHdC    


In [40]:
# To switch to another token (by name) again use auth_switch() or the cli
# There's no command for printing the current selection. Instead, a * is placed on the left

# run huggingface-cli auth switch --help for other options
auth_switch(token_name="fine-grained-token")
print(auth_list())

!huggingface-cli auth switch --token-name read-token

  name                | token          
----------------------|---------------
* fine-grained-token  | hf_****KBKQ    
  read-token          | hf_****nHdC    
None
Your token has been saved to /Users/christos/.cache/huggingface/token
The current active token is: read-token


## Repository Management

In [ ]:
from huggingface_hub import HfApi, ModelInfo
from huggingface_hub import list_models

type(next(list_models(model_name="deepseek-ai/DeepSeek-R1-0528-Qwen3-8B")))
print(ModelInfo(id="deepseek-ai/DeepSeek-R1-0528-Qwen3-8B").json)

ModelInfo(id='deepseek-ai/DeepSeek-R1-0528-Qwen3-8B', author=None, sha=None, created_at=None, last_modified=None, private=None, disabled=None, downloads=None, downloads_all_time=None, gated=None, gguf=None, inference=None, inference_provider_mapping=None, likes=None, library_name=None, tags=None, pipeline_tag=None, mask_token=None, card_data=None, widget_data=None, model_index=None, config=None, transformers_info=None, trending_score=None, siblings=None, spaces=None, safetensors=None, security_repo_status=None, xet_enabled=None)


In [ ]:
# Download an HF repo (model) snapshot
from huggingface_hub import hf_hub_download, snapshot_download
from os.path import expanduser, join

# Pick a relatively small model for convenience. The savings in large models are huge!
model = "amazon/chronos-t5-small"

In [ ]:
# Example: download a small model to a custom dir
snapshot_download(
    repo_id=model,
    cache_dir=expanduser("~/.cache/chris/all")
)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

main-figure.png:   0%|          | 0.00/232k [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

'/Users/christos/.cache/chris/all/models--amazon--chronos-t5-small/snapshots/6bcab32367f856115ca07bdd7f4956eced00a4b5'

In [12]:
snapshot_download(
    repo_id=model,
    cache_dir=expanduser("~/.cache/chris/main")
)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

config.json: 0.00B [00:00, ?B/s]

main-figure.png:   0%|          | 0.00/232k [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

'/Users/christos/.cache/chris/main/models--amazon--chronos-t5-small/snapshots/6bcab32367f856115ca07bdd7f4956eced00a4b5'

### Download size comparison

Eventualy, the example was nasty because the model repo contains one version only.  

We can check this from the "Files and Versions" tab in the model card.  

Generally, it is not easy to find small models with multiple versions. LLMs on the contrary do contain multiple versions.

In [15]:
%%bash
cd ~/.cache/chris
du -d1 -h .

182M	./all
182M	./main
364M	.


### Test `local_dir` and `cache_dir`. 

When `local-dir` is present, `cache-dir` does not contain actual data, only some partial metadata info (of unspecified nature: HF docs does not provide info).  

According to the [HF doc](https://huggingface.co/docs/huggingface_hub/v0.33.1/en/package_reference/hf_api#huggingface_hub.HfApi.snapshot_download) when `local_dir` is present

the function creates `<local_dir>/.cache/huggingface` with __metadata-only info__. The actual data are stored in the other directories under `local-dir`, in the SAME STRUCTURE as  

under the `snapshots` dirs in the huggingface model caches.

In [16]:
snapshot_download(
    repo_id=model,
    local_dir=expanduser("~/.cache/chris/local-dir"),
    cache_dir=expanduser("~/.cache/chris/cache-dir")
)

Fetching 6 files:   0%|          | 0/6 [00:00<?, ?it/s]

main-figure.png:   0%|          | 0.00/232k [00:00<?, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

README.md: 0.00B [00:00, ?B/s]

generation_config.json:   0%|          | 0.00/142 [00:00<?, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/185M [00:00<?, ?B/s]

'/Users/christos/.cache/chris/local-dir'

In [17]:
%%bash
find ~/.cache/chris/local-dir/.cache

/Users/christos/.cache/chris/local-dir/.cache
/Users/christos/.cache/chris/local-dir/.cache/huggingface
/Users/christos/.cache/chris/local-dir/.cache/huggingface/.gitignore
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/model.safetensors.metadata
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/.gitattributes.metadata
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/config.json.metadata
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/config.json.lock
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/generation_config.json.lock
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/.gitattributes.lock
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/generation_config.json.metadata
/Users/christos/.cache/chris/local-dir/.cache/huggingface/download/figures
/Users/christos/.cache/chris/local-dir/.cache/h

In [51]:
%%bash

# This is important! find requires relative paths to operate. E.g. we cannot write 
# $ find ~/.cache/chris -path "local-dir/.cache" -prune -o
cd ~/.cache/chris/local-dir
find . -path './.cache' -prune -o -print

.
./model.safetensors
./config.json
./generation_config.json
./README.md
./figures
./figures/main-figure.png
./.gitattributes
